# Colab Pro LLM Backend (Ollama + Pinggy.io)
Run this notebook on a Google Colab Pro instance with a hardware accelerator (T4, V100, or A100 GPU) enabled.
It sets up an OpenAI-compatible API endpoint exposing an Ollama model using Pinggy.io to bypass network firewalls.

In [ ]:
# 1. Install require dependency for Ollama on new Ubuntu versions
!apt-get update -y && apt-get install -y zstd

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import os
import subprocess
import time

# 3. Tell Ollama to accept connections from anyone
os.environ["OLLAMA_HOST"] = "0.0.0.0"

# 4. Kill any stuck processes from previous runs
subprocess.run(["pkill", "-f", "ollama"])
time.sleep(2)

# 5. Start Ollama Server in the background
ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
print("✅ Ollama Server started in background.")

# 6. Give Ollama a few seconds to start up before pulling the model
time.sleep(3)

# 7. Pull the desired Model (We default to mistral, but can be llama3, etc.)
MODEL_NAME = "mistral" # YOU CAN CHANGE THIS
print(f"⏳ Pulling {MODEL_NAME}... this may take a minute or two depending on model size.")
subprocess.run(["/usr/local/bin/ollama", "pull", MODEL_NAME])
print(f"✅ Model {MODEL_NAME} pulled successfully.")

# 8. Start Pinggy Tunnel (Most reliable fallback)
print("\n" + "="*60)
print("⏳ Starting Pinggy Tunnel...")
pinggy_process = subprocess.Popen(
    ["ssh", "-p", "443", "-R0:localhost:11434", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "a.pinggy.io"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5) # Give it 5 seconds to establish the SSH handshake

url_found = False
for _ in range(20):
    line = pinggy_process.stdout.readline()
    if "http://" in line or "https://" in line:
        if "pinggy.link" in line:
            public_url = line.strip()
            if public_url.startswith("http://"):
                public_url = "https://" + public_url[7:] # Force HTTPS display
            print(f"🔥 PUBLIC TUNNEL URL: {public_url}")
            print("="*60 + "\n")
            print(f"👉 Copy the URL above and use `{public_url}/v1` as your base_url in your local IDE.")
            url_found = True
            break

if not url_found:
    print("❌ Could not extract Tunnel URL automatically. Check Colab logs for errors.")


## Keeping the Notebook Alive
Run this cell to prevent Colab from disconnecting while you use the API from your local IDE.

In [ ]:
import time
print("Entering infinite loop to keep instance alive...")
print("Press Stop to terminate.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStopped.")